# MolGAN: an implicit generative model for small molecular graphs
## A GPU-oriented reproduction of De Cao & Kipf (2018)

This notebook explains and reproduces **MolGAN: An implicit generative model for small molecular graphs**.

- [Paper](https://arxiv.org/abs/1805.11973)
- [Authors' repository](https://github.com/nicola-decao/MolGAN)

The implementation trains directly on QM9 heavy-atom graphs, uses a one-shot Gumbel-softmax generator, a relational graph discriminator with WGAN-GP, and a learned reward network for QED-directed generation. Inputs and expensive RDKit caches are stored under `data/MolGAN/`; reusable weights are stored under `models/MolGAN/`.

## 1. Why MolGAN is implicit

MolGAN does not define a tractable likelihood and has no encoder. A generator maps noise $z\sim\mathcal N(0,I)$ directly to a complete probabilistic graph:

$$G_\theta(z)=(\tilde X,\tilde A),$$

where $\tilde X$ contains node-type logits and $\tilde A$ contains edge-type logits. Gumbel-softmax provides differentiable samples from these categorical distributions. A discriminator assigns high scores to real QM9 graphs and low scores to generated graphs.

Because all nodes and edges are emitted simultaneously, generation is fast and permutation choices are learned implicitly—but chemical constraints are not guaranteed.

## 2. Adversarial and reinforcement objectives

We use the stable Wasserstein critic with gradient penalty:

$$L_D=\mathbb E[D(G_{fake})]-\mathbb E[D(G_{real})]+\lambda_{GP}L_{GP},\qquad L_{adv}=-\mathbb E[D(G_{fake})].$$

MolGAN additionally learns a differentiable reward surrogate $R_\psi(G)$. RDKit supplies non-differentiable molecular rewards; the surrogate regresses those values, allowing generator gradients:

$$L_G=\lambda L_{adv}-(1-\lambda)\mathbb E[R_\psi(G_{fake})].$$

Here the reward is sanitized-molecule QED, with invalid graphs receiving zero. The paper explores property optimization and highlights a serious failure mode: reward optimization can intensify mode collapse, producing high-scoring but repetitive molecules.

In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import Draw, QED
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.datasets import QM9

SEED=31
random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA=device.type=='cuda'
if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
    torch.set_float32_matmul_precision('high')
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='MolGAN' else Path.cwd()
DATA_DIR=PROJECT_ROOT/'data'/'MolGAN'; MODEL_DIR=PROJECT_ROOT/'models'/'MolGAN'
DATA_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True)
print(f'PyTorch={torch.__version__} device={device}')
if USE_CUDA:
    p=torch.cuda.get_device_properties(0); print(f'{p.name}, {p.total_memory/2**30:.1f} GiB, TF32 enabled')

## 3. Fixed-size QM9 molecular graphs

Hydrogens are removed and molecules are padded to nine heavy atoms. Node classes are `PAD, C, N, O, F`; edge classes are `NONE, SINGLE, DOUBLE, TRIPLE, AROMATIC`. Only upper-triangular edge decisions are generated and then mirrored, guaranteeing undirected adjacency. The converted tensor dataset is cached once.

In [ ]:
MAX_NODES=9; ATOM_NUMBERS=[0,6,7,8,9]; ATOM_NAMES=['PAD','C','N','O','F']
BOND_NAMES=['NONE','SINGLE','DOUBLE','TRIPLE','AROMATIC']
N_NODE_TYPES,N_EDGE_TYPES=len(ATOM_NAMES),len(BOND_NAMES)
GRAPH_CACHE=DATA_DIR/'qm9_graphs.pt'
def build_graph_cache():
    dataset=QM9(root=PROJECT_ROOT/'data'/'QM9')
    nodes=torch.zeros((len(dataset),MAX_NODES),dtype=torch.long)
    edges=torch.zeros((len(dataset),MAX_NODES,MAX_NODES),dtype=torch.long)
    atom_class={z:i for i,z in enumerate(ATOM_NUMBERS)}
    for index,graph in enumerate(dataset):
        heavy=torch.where(graph.z>1)[0]; old_to_new=torch.full((graph.num_nodes,),-1,dtype=torch.long)
        old_to_new[heavy]=torch.arange(len(heavy))
        nodes[index,:len(heavy)]=torch.tensor([atom_class[int(z)] for z in graph.z[heavy]])
        src,dst=graph.edge_index; keep=(old_to_new[src]>=0)&(old_to_new[dst]>=0)
        edges[index,old_to_new[src[keep]],old_to_new[dst[keep]]]=graph.edge_attr[keep,:4].argmax(-1)+1
        if (index+1)%25000==0: print(f'Converted {index+1:,}/{len(dataset):,}')
    payload={'nodes':nodes,'edges':edges,'version':1,'atom_numbers':ATOM_NUMBERS}
    torch.save(payload,GRAPH_CACHE); return payload
cache=torch.load(GRAPH_CACHE,weights_only=True) if GRAPH_CACHE.exists() else build_graph_cache()
if cache['atom_numbers']!=ATOM_NUMBERS: raise ValueError('Incompatible graph cache.')
all_nodes,all_edges=cache['nodes'],cache['edges']
print(f'Graphs={len(all_nodes):,}; cache={GRAPH_CACHE}')

## 4. RDKit conversion and persistent QED rewards

The reward cache stores QED values and canonical SMILES for all training graphs. This is CPU-heavy but runs only once. The same converter is later used to evaluate generated graphs; sanitization failures receive reward zero.

In [ ]:
BOND_TYPES={1:Chem.BondType.SINGLE,2:Chem.BondType.DOUBLE,3:Chem.BondType.TRIPLE,4:Chem.BondType.AROMATIC}
def tensor_to_mol(node_types,edge_types):
    rw=Chem.RWMol(); present=torch.where(node_types>0)[0].tolist(); mapping={}
    if not present: return None
    for old in present: mapping[old]=rw.AddAtom(Chem.Atom(ATOM_NUMBERS[int(node_types[old])]))
    try:
        for ai,i in enumerate(present):
            for j in present[ai+1:]:
                bond=int(edge_types[i,j])
                if bond:
                    rw.AddBond(mapping[i],mapping[j],BOND_TYPES[bond])
                    if bond==4: rw.GetAtomWithIdx(mapping[i]).SetIsAromatic(True); rw.GetAtomWithIdx(mapping[j]).SetIsAromatic(True)
        mol=rw.GetMol(); Chem.SanitizeMol(mol); return mol
    except Exception: return None
REWARD_CACHE=DATA_DIR/'qm9_qed_rewards.pt'
if REWARD_CACHE.exists():
    reward_data=torch.load(REWARD_CACHE,weights_only=False)
else:
    rewards=torch.zeros(len(all_nodes)); canonical=[]
    for i,(nodes,edges) in enumerate(zip(all_nodes,all_edges)):
        mol=tensor_to_mol(nodes,edges)
        canonical.append(None if mol is None else Chem.MolToSmiles(mol)); rewards[i]=0 if mol is None else QED.qed(mol)
        if (i+1)%25000==0: print(f'Rewarded {i+1:,}/{len(all_nodes):,}')
    reward_data={'qed':rewards,'canonical':canonical,'version':1}; torch.save(reward_data,REWARD_CACHE)
qed_targets=reward_data['qed']; training_smiles={s for s in reward_data['canonical'] if s}
print(f'Mean QED={qed_targets.mean():.3f}; canonical training molecules={len(training_smiles):,}')

In [ ]:
BATCH_SIZE=512 if USE_CUDA else 64; NUM_WORKERS=min(8,os.cpu_count() or 1) if USE_CUDA else 0
loader=DataLoader(TensorDataset(all_nodes,all_edges,qed_targets),batch_size=BATCH_SIZE,shuffle=True,
                  num_workers=NUM_WORKERS,pin_memory=USE_CUDA,persistent_workers=NUM_WORKERS>0,drop_last=True)
print(f'batch={BATCH_SIZE}; workers={NUM_WORKERS}; steps={len(loader)}')

## 5. Generator, relational critic, and reward network

The generator emits categorical logits for all node slots and upper-triangular edges. The critic and reward network use bond-type-specific dense message passing over soft graphs, so gradients can flow through Gumbel-softmax samples. The critic's graph representation uses gated pooling over non-padding probability.

In [ ]:
LATENT_DIM=64; HIDDEN=256
class Generator(nn.Module):
    def __init__(self):
        super().__init__(); upper=torch.triu_indices(MAX_NODES,MAX_NODES,1)
        self.register_buffer('row',upper[0]); self.register_buffer('col',upper[1])
        output=MAX_NODES*N_NODE_TYPES+upper.shape[1]*N_EDGE_TYPES
        self.net=nn.Sequential(nn.Linear(LATENT_DIM,HIDDEN),nn.Tanh(),nn.Linear(HIDDEN,512),nn.Tanh(),nn.Linear(512,output))
    def forward(self,z,tau=1.,hard=False):
        raw=self.net(z); split=MAX_NODES*N_NODE_TYPES
        node_logits=raw[:,:split].view(-1,MAX_NODES,N_NODE_TYPES)
        upper_logits=raw[:,split:].view(-1,len(self.row),N_EDGE_TYPES)
        nodes=F.gumbel_softmax(node_logits,tau=tau,hard=hard,dim=-1)
        upper=F.gumbel_softmax(upper_logits,tau=tau,hard=hard,dim=-1)
        edges=raw.new_zeros((len(z),MAX_NODES,MAX_NODES,N_EDGE_TYPES))
        edges[:,self.row,self.col]=upper; edges[:,self.col,self.row]=upper
        edges[:,torch.arange(MAX_NODES),torch.arange(MAX_NODES),0]=1
        return nodes,edges
class RelationalGraphNet(nn.Module):
    def __init__(self,out_dim):
        super().__init__(); self.input=nn.Linear(N_NODE_TYPES,128)
        self.self_layers=nn.ModuleList([nn.Linear(128,128) for _ in range(3)])
        self.rel_layers=nn.ModuleList([nn.ModuleList([nn.Linear(128,128,bias=False) for _ in range(N_EDGE_TYPES-1)]) for _ in range(3)])
        self.norms=nn.ModuleList([nn.LayerNorm(128) for _ in range(3)])
        self.gate=nn.Linear(128,1); self.output=nn.Sequential(nn.Linear(128,128),nn.Tanh(),nn.Linear(128,out_dim))
    def forward(self,nodes,edges):
        h=F.leaky_relu(self.input(nodes),.1)
        for self_layer,relations,norm in zip(self.self_layers,self.rel_layers,self.norms):
            message=self_layer(h)
            for bond,layer in enumerate(relations,start=1): message=message+torch.bmm(edges[...,bond],layer(h))
            h=norm(F.leaky_relu(message,.1))
        present=1-nodes[...,0]; gate=torch.sigmoid(self.gate(h).squeeze(-1))*present
        pooled=(h*gate[...,None]).sum(1)/gate.sum(1,keepdim=True).clamp_min(1e-6)
        return self.output(pooled).squeeze(-1)
generator=Generator().to(device); critic=RelationalGraphNet(1).to(device); reward_net=RelationalGraphNet(1).to(device)
print(f'G={sum(p.numel() for p in generator.parameters()):,}; D={sum(p.numel() for p in critic.parameters()):,}; R={sum(p.numel() for p in reward_net.parameters()):,}')

In [ ]:
def as_one_hot(nodes,edges): return F.one_hot(nodes,N_NODE_TYPES).float(),F.one_hot(edges,N_EDGE_TYPES).float()
def gradient_penalty(real_n,real_e,fake_n,fake_e):
    alpha=torch.rand(len(real_n),1,1,device=device); n=(alpha*real_n+(1-alpha)*fake_n).requires_grad_(True)
    alpha_e=alpha.unsqueeze(-1); e=(alpha_e*real_e+(1-alpha_e)*fake_e).requires_grad_(True)
    score=critic(n,e); gn,ge=torch.autograd.grad(score,(n,e),torch.ones_like(score),create_graph=True)
    norm=torch.cat([gn.flatten(1),ge.flatten(1)],1).norm(2,dim=1)
    return ((norm-1)**2).mean()
@torch.no_grad()
def hard_graphs(n=256,tau=.5):
    nodes,edges=generator(torch.randn(n,LATENT_DIM,device=device),tau,True)
    node_ids=nodes.argmax(-1).cpu(); edge_ids=edges.argmax(-1).cpu()
    present=node_ids.ne(0); edge_ids=edge_ids*present[:,:,None]*present[:,None,:]
    return node_ids,edge_ids
def label_generated(nodes,edges):
    rewards=[]
    for n,e in zip(nodes,edges):
        mol=tensor_to_mol(n,e); rewards.append(0. if mol is None else QED.qed(mol))
    return torch.tensor(rewards,dtype=torch.float32)
@torch.no_grad()
def sample_metrics(n=1000):
    nodes,edges=hard_graphs(n); smiles=[]; rewards=[]
    for x,a in zip(nodes,edges):
        mol=tensor_to_mol(x,a)
        if mol is not None: smiles.append(Chem.MolToSmiles(mol)); rewards.append(QED.qed(mol))
    unique=set(smiles)
    return {'validity':len(smiles)/n,'uniqueness':len(unique)/max(1,len(smiles)),
            'novelty':len(unique-training_smiles)/max(1,len(unique)),'mean_qed':sum(rewards)/max(1,len(rewards))}

## 6. Cluster training

WGAN gradient penalties require second-order gradients and are kept in float32 for numerical stability; TF32 still accelerates matrix multiplications on supported GPUs. The critic and QED surrogate update every batch; the generator updates every five critic steps. Generated graphs periodically receive fresh RDKit labels to reduce reward-model extrapolation. The best validation score combines validity and uniqueness rather than QED alone, resisting trivial reward collapse.

In [ ]:
EPOCHS,N_CRITIC,EVAL_EVERY,FORCE_RETRAIN=100,5,5,False
GP_WEIGHT,ADV_WEIGHT=10.,.8; CHECKPOINT=MODEL_DIR/'molgan_qm9.pt'
CONFIG={'latent':LATENT_DIM,'max_nodes':MAX_NODES,'atom_numbers':ATOM_NUMBERS,'seed':SEED,'cache_version':cache['version']}
history={'d':[],'g':[],'reward':[],'score':[]}; best_score=-1.; start=time.time()
if CHECKPOINT.exists() and not FORCE_RETRAIN:
    ckpt=torch.load(CHECKPOINT,map_location=device,weights_only=True)
    if ckpt['config']!=CONFIG: raise ValueError('Incompatible checkpoint; set FORCE_RETRAIN=True.')
    generator.load_state_dict(ckpt['generator']); critic.load_state_dict(ckpt['critic']); reward_net.load_state_dict(ckpt['reward_net']); best_score=ckpt['best_score']
    print('Loaded',CHECKPOINT)
else:
    opt_g=torch.optim.Adam(generator.parameters(),lr=1e-4,betas=(0.5,.9),fused=USE_CUDA)
    opt_d=torch.optim.Adam(critic.parameters(),lr=1e-4,betas=(0.5,.9),fused=USE_CUDA)
    opt_r=torch.optim.Adam(reward_net.parameters(),lr=2e-4,fused=USE_CUDA)
    global_step=0
    for epoch in range(1,EPOCHS+1):
        generator.train(); critic.train(); reward_net.train(); sums={'d':0.,'g':0.,'r':0.}; batches=0
        tau=max(.35,1.0*.98**epoch)
        for node_ids,edge_ids,reward in loader:
            node_ids,edge_ids,reward=node_ids.to(device,non_blocking=True),edge_ids.to(device,non_blocking=True),reward.to(device,non_blocking=True)
            real_n,real_e=as_one_hot(node_ids,edge_ids); z=torch.randn(len(node_ids),LATENT_DIM,device=device)
            fake_n,fake_e=generator(z,tau,False); opt_d.zero_grad(set_to_none=True)
            d_loss=critic(fake_n.detach(),fake_e.detach()).mean()-critic(real_n,real_e).mean()+GP_WEIGHT*gradient_penalty(real_n,real_e,fake_n.detach(),fake_e.detach())
            d_loss.backward(); opt_d.step()
            opt_r.zero_grad(set_to_none=True); r_loss=F.mse_loss(reward_net(real_n,real_e).sigmoid(),reward)
            r_loss.backward(); opt_r.step(); g_value=0.
            if global_step%N_CRITIC==0:
                opt_g.zero_grad(set_to_none=True); fake_n,fake_e=generator(torch.randn(len(node_ids),LATENT_DIM,device=device),tau,False)
                g_loss=ADV_WEIGHT*(-critic(fake_n,fake_e).mean())-(1-ADV_WEIGHT)*reward_net(fake_n,fake_e).sigmoid().mean()
                g_loss.backward(); nn.utils.clip_grad_norm_(generator.parameters(),10.); opt_g.step(); g_value=g_loss.item()
            if global_step%50==0:
                hn,he=hard_graphs(min(128,len(node_ids)),tau); target=label_generated(hn,he).to(device)
                fn,fe=F.one_hot(hn.to(device),N_NODE_TYPES).float(),F.one_hot(he.to(device),N_EDGE_TYPES).float()
                opt_r.zero_grad(set_to_none=True); extra=F.mse_loss(reward_net(fn,fe).sigmoid(),target); extra.backward(); opt_r.step()
            sums['d']+=d_loss.item(); sums['g']+=g_value; sums['r']+=r_loss.item(); batches+=1; global_step+=1
        for key in ('d','g','r'): history[{'r':'reward'}.get(key,key)].append(sums[key]/batches)
        if epoch%EVAL_EVERY==0:
            generator.eval(); metrics=sample_metrics(1000); score=metrics['validity']*metrics['uniqueness']; history['score'].append(score)
            print(f'{epoch:03d} D={sums["d"]/batches:.3f} G={sums["g"]/batches:.3f} R={sums["r"]/batches:.4f}',metrics)
            if score>best_score:
                best_score=score; torch.save({'generator':generator.state_dict(),'critic':critic.state_dict(),'reward_net':reward_net.state_dict(),'config':CONFIG,'best_score':best_score},CHECKPOINT)
    ckpt=torch.load(CHECKPOINT,map_location=device,weights_only=True); generator.load_state_dict(ckpt['generator'])
generator.eval(); final_metrics=sample_metrics(5000)
print('Final:',json.dumps(final_metrics,indent=2),f'elapsed={(time.time()-start)/60:.1f} min')

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
axes[0].plot(history['d']); axes[0].set_title('Critic loss')
axes[1].plot(history['g']); axes[1].set_title('Generator loss')
axes[2].plot(history['reward']); axes[2].set_title('Reward regression loss')
for ax in axes: ax.set_xlabel('epoch')
plt.tight_layout(); plt.show()

In [ ]:
nodes,edges=hard_graphs(64,.35); molecules=[]
for x,a in zip(nodes,edges):
    mol=tensor_to_mol(x,a)
    if mol is not None and Chem.MolToSmiles(mol) not in {Chem.MolToSmiles(m) for m in molecules}: molecules.append(mol)
    if len(molecules)==20: break
if molecules:
    display(Draw.MolsToGridImage(molecules,molsPerRow=5,subImgSize=(220,180),legends=[f'QED {QED.qed(m):.2f}' for m in molecules]))
else: print('No valid molecules in this sample.')

## 7. What was reproduced—and what to watch

### Reproduced

- direct non-autoregressive generation of node and edge tensors;
- differentiable categorical sampling with Gumbel-softmax;
- relational graph discrimination and Wasserstein adversarial training;
- a learned surrogate for non-differentiable molecular rewards;
- QM9 validity, uniqueness, novelty, QED, and visual inspection.

### Differences and limitations

- Network sizes, reward schedule, and WGAN-GP details are modern reproduction choices rather than exact legacy TensorFlow settings.
- QED is used as the property reward; other paper experiments use different objectives.
- RDKit sanitization defines validity here. Validity does not imply synthesizability, safety, novelty of scaffold, or useful activity.
- No likelihood is available, and GAN losses do not reliably diagnose convergence.
- MolGAN is especially vulnerable to mode collapse; always report uniqueness and novelty beside reward.
- The fixed nine-heavy-atom canvas cannot scale to drug-sized molecules.

For a rigorous study, train multiple seeds, report distributions and confidence intervals, compare reward-free and reward-directed runs, measure scaffold diversity, evaluate synthetic accessibility, and retain a fixed held-out reference set for distributional comparisons.

## Reference

De Cao N, Kipf T. *MolGAN: An implicit generative model for small molecular graphs.* arXiv:1805.11973, 2018.